# Lab 8E: First-Order Logic Workbench — ASKVARS, Ontologies, and Limits

## Learning goals
By the end of this lab, students can:
- use a finite-domain workbench to answer first-order queries with variables;
- explain how ontology choices affect what a KB can express easily;
- visualize the scaling advantage of first-order schemas over propositional ground instances;
- identify a key limitation of crisp first-order predicates: vague categories depend on interpretation.

## Chapter 8 connection
Chapter 8 emphasizes that a useful representation language should be declarative, compositional, expressive, context independent, and unambiguous. It also warns that both propositional and first-order logic struggle with vague predicates. This lab gives students a capstone workbench for those ideas.


In [ ]:
# Run this cell first.

from itertools import product
from html import escape
from collections import defaultdict
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False
    print("ipywidgets is not installed. Static demonstrations will be shown instead.")


## 1. A small family ontology

We choose objects, predicates, and definitions. This is a miniature version of Chapter 8's knowledge-engineering process.

**Domain objects:** people.  
**Base predicate:** `Parent(x,y)`.  
**Defined predicates:**

- `Grandparent(x,z) ⇔ ∃y Parent(x,y) ∧ Parent(y,z)`
- `Sibling(x,y) ⇔ x ≠ y ∧ ∃p Parent(p,x) ∧ Parent(p,y)`
- `Ancestor(x,z)` is the transitive closure of `Parent` in this finite workbench.


In [ ]:
PEOPLE = ["Alice", "Bob", "Cara", "Dinesh", "Elena", "Finn"]
PARENT = {
    ("Alice", "Bob"),
    ("Alice", "Cara"),
    ("Bob", "Dinesh"),
    ("Cara", "Elena"),
    ("Dinesh", "Finn"),
}
HEIGHT_CM = {
    "Alice": 165,
    "Bob": 178,
    "Cara": 171,
    "Dinesh": 183,
    "Elena": 160,
    "Finn": 169,
}


def parent(x, y):
    return (x, y) in PARENT


def grandparent(x, z):
    return any(parent(x, y) and parent(y, z) for y in PEOPLE)


def sibling(x, y):
    return x != y and any(parent(p, x) and parent(p, y) for p in PEOPLE)


def ancestor(x, z):
    frontier = [z]
    seen = set()
    # Reverse traversal: find parents of current node until x is found.
    while frontier:
        child = frontier.pop()
        for p, c in PARENT:
            if c == child and p not in seen:
                if p == x:
                    return True
                seen.add(p)
                frontier.append(p)
    return False

PREDICATES = {
    "Parent(x,y)": parent,
    "Grandparent(x,y)": grandparent,
    "Sibling(x,y)": sibling,
    "Ancestor(x,y)": ancestor,
}


def askvars(predicate_name, var_names=("x", "y")):
    pred = PREDICATES[predicate_name]
    answers = []
    for values in product(PEOPLE, repeat=len(var_names)):
        if pred(*values):
            answers.append(dict(zip(var_names, values)))
    return answers

for q in PREDICATES:
    print(q, "->", askvars(q)[:6], "... total", len(askvars(q)))


In [ ]:
def draw_family(highlight_pairs=None, title="Family relation graph"):
    highlight_pairs = set(highlight_pairs or [])
    pos = {
        "Alice": (3, 3),
        "Bob": (1.8, 2),
        "Cara": (4.2, 2),
        "Dinesh": (1.8, 1),
        "Elena": (4.2, 1),
        "Finn": (1.8, 0),
    }
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.set_title(title)
    ax.axis("off")
    for p, c in PARENT:
        ax.annotate("", xy=pos[c], xytext=pos[p], arrowprops=dict(arrowstyle="->", lw=1.7))
        mx, my = (pos[p][0] + pos[c][0]) / 2, (pos[p][1] + pos[c][1]) / 2
        ax.text(mx, my, "Parent", fontsize=8, ha="center")
    for person, (x, y) in pos.items():
        ax.add_patch(plt.Circle((x, y), 0.18, fill=False, lw=2))
        ax.text(x, y, person, ha="center", va="center")
    # Highlight query answers with curved dashed arrows.
    for a, b in highlight_pairs:
        if a in pos and b in pos:
            ax.annotate("", xy=pos[b], xytext=pos[a],
                        arrowprops=dict(arrowstyle="->", lw=2.2, linestyle="--", connectionstyle="arc3,rad=0.2"))
    ax.set_xlim(0.8, 5.2)
    ax.set_ylim(-0.4, 3.5)
    plt.show()


def display_answers(predicate_name):
    answers = askvars(predicate_name)
    pairs = [(a["x"], a["y"]) for a in answers]
    draw_family(pairs, f"ASKVARS query: {predicate_name}")
    html = f"<h3>Substitutions for {escape(predicate_name)}</h3>"
    html += "<table><tr><th>x</th><th>y</th></tr>"
    for ans in answers:
        html += f"<tr><td>{ans['x']}</td><td>{ans['y']}</td></tr>"
    html += "</table>"
    display(HTML(html))


display_answers("Grandparent(x,y)")


In [ ]:
if WIDGETS_AVAILABLE:
    dropdown = widgets.Dropdown(options=list(PREDICATES.keys()), value="Grandparent(x,y)", description="Query:", layout=widgets.Layout(width="50%"))
    out = widgets.Output()
    def refresh(change=None):
        with out:
            clear_output(wait=True)
            display_answers(dropdown.value)
    dropdown.observe(refresh, names="value")
    display(dropdown, out)
    refresh()
else:
    display_answers("Ancestor(x,y)")


## 2. Representation scaling: propositional symbols versus FOL schemas

Suppose we need to talk about `Parent(a,b)` and `Ancestor(a,b)` for many people. A propositional representation needs many separate ground symbols or rules. FOL keeps one predicate symbol and quantified schemas.


In [ ]:
def scaling_counts(n):
    parent_symbols = n * n
    ancestor_ground_rules = n * n  # Parent(a,b) => Ancestor(a,b), one instance per pair if grounded.
    transitive_ground_rules = n * n * n  # Ancestor(a,b) and Ancestor(b,c) => Ancestor(a,c).
    fol_schemas = 3  # Parent, base ancestor schema, transitive schema.
    return parent_symbols, ancestor_ground_rules + transitive_ground_rules, fol_schemas

sizes = list(range(2, 61))
parent_counts = []
rule_counts = []
fol_counts = []
for n in sizes:
    p, r, f = scaling_counts(n)
    parent_counts.append(p)
    rule_counts.append(r)
    fol_counts.append(f)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sizes, parent_counts, label="ground Parent(a,b) symbols")
ax.plot(sizes, rule_counts, label="ground ancestor rule instances")
ax.plot(sizes, fol_counts, label="FOL schemas")
ax.set_xlabel("Number of people")
ax.set_ylabel("Representation count")
ax.set_title("FOL captures repeated relational structure compactly")
ax.legend()
plt.show()

print("For 60 people:")
print("Parent ground symbols:", parent_counts[-1])
print("Ancestor-related ground rule instances:", rule_counts[-1])
print("FOL schemas:", fol_counts[-1])


## 3. Ontology choice changes what is easy to ask

Two possible ontologies for a classroom domain:

**Ontology A: unary predicate**
- `Student(x)`, `Course(c)`, `Enrolled(x,c)`

**Ontology B: enrollment events as objects**
- `Enrollment(e)`, `StudentOf(e)=x`, `CourseOf(e)=c`, `TermOf(e)=t`, `GradeOf(e)=g`

Ontology A is compact for current enrollment. Ontology B is better when each enrollment has properties such as term and grade.


In [ ]:
STUDENTS = ["Ana", "Bo", "Chen"]
COURSES = ["AI", "Databases"]
ENROLLED = {("Ana", "AI"), ("Bo", "AI"), ("Chen", "Databases")}
ENROLLMENTS = {
    "E1": {"student": "Ana", "course": "AI", "term": "Fall", "grade": "A"},
    "E2": {"student": "Bo", "course": "AI", "term": "Fall", "grade": "B"},
    "E3": {"student": "Chen", "course": "Databases", "term": "Spring", "grade": "A"},
}


def draw_enrollment_ontologies():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ax = axes[0]
    ax.set_title("Ontology A: Enrolled(student, course)")
    ax.axis("off")
    spos = {s: (0.2, 0.8 - i*0.25) for i, s in enumerate(STUDENTS)}
    cpos = {c: (0.8, 0.7 - i*0.35) for i, c in enumerate(COURSES)}
    for s, p in spos.items():
        ax.text(*p, s, ha="center", va="center", bbox=dict(boxstyle="circle", fill=False))
    for c, p in cpos.items():
        ax.text(*p, c, ha="center", va="center", bbox=dict(boxstyle="round", fill=False))
    for s, c in ENROLLED:
        ax.annotate("", xy=cpos[c], xytext=spos[s], arrowprops=dict(arrowstyle="->"))
        mx = (spos[s][0]+cpos[c][0])/2; my=(spos[s][1]+cpos[c][1])/2
        ax.text(mx, my, "Enrolled", fontsize=8)

    ax = axes[1]
    ax.set_title("Ontology B: enrollment events as objects")
    ax.axis("off")
    center_x = 0.5
    for i, (e, data) in enumerate(ENROLLMENTS.items()):
        y = 0.85 - i*0.28
        ax.text(center_x, y, e, ha="center", va="center", bbox=dict(boxstyle="circle", fill=False))
        ax.text(0.08, y, data["student"], ha="center", va="center", bbox=dict(boxstyle="round", fill=False))
        ax.text(0.92, y, data["course"], ha="center", va="center", bbox=dict(boxstyle="round", fill=False))
        ax.text(0.5, y-0.09, f"term={data['term']}, grade={data['grade']}", ha="center", fontsize=9)
        ax.annotate("", xy=(center_x-0.06, y), xytext=(0.14, y), arrowprops=dict(arrowstyle="->"))
        ax.annotate("", xy=(center_x+0.06, y), xytext=(0.86, y), arrowprops=dict(arrowstyle="<-"))
    plt.show()


draw_enrollment_ontologies()

print("Query easy in Ontology A: Who is enrolled in AI?", [s for s, c in ENROLLED if c == "AI"])
print("Query easy in Ontology B: Who earned A in Fall?", [d['student'] for d in ENROLLMENTS.values() if d['grade'] == 'A' and d['term'] == 'Fall'])


## 4. A limitation: vague predicates need interpretation choices

`Tall(x)` looks crisp in logic, but real categories can be vague. The truth of `Tall(Bob)` below depends on the threshold chosen by the model designer.


In [ ]:
def draw_tall_threshold(threshold=175):
    people = list(HEIGHT_CM.keys())
    heights = [HEIGHT_CM[p] for p in people]
    tall = [h >= threshold for h in heights]
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(people, heights)
    ax.axhline(threshold, linestyle="--", label=f"Tall threshold = {threshold} cm")
    ax.set_ylabel("Height in cm")
    ax.set_title("Same predicate name, different interpretation threshold")
    ax.legend()
    for bar, is_tall in zip(bars, tall):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, "Tall" if is_tall else "not Tall", ha="center", fontsize=9)
    plt.show()
    display(HTML("<h3>Truth values</h3><ul>" + "".join(f"<li>Tall({p}) = {h >= threshold}</li>" for p, h in HEIGHT_CM.items()) + "</ul>"))

if WIDGETS_AVAILABLE:
    slider = widgets.IntSlider(value=175, min=155, max=190, step=1, description="threshold")
    out = widgets.Output()
    def update(change=None):
        with out:
            clear_output(wait=True)
            draw_tall_threshold(slider.value)
    slider.observe(update, names="value")
    display(slider, out)
    update()
else:
    draw_tall_threshold(175)


## Student practice

1. Add another person and parent fact. Which ASKVARS results change?
2. Create a new defined predicate `Cousin(x,y)` and visualize its answers.
3. Explain why `Tall(x)` is harder to represent with classical logic than `Parent(x,y)`.
4. Pick one ontology choice in the enrollment example and defend it for a specific query workload.
